<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: STL residuals

**[STL residuals](https://learning.nextia-ai.com/courses/anomaly-detection/m05/stl-residuals/)** · Anomaly Detection: Find What Is Unusual · Module 5, lesson 2 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** split a series into trend, season and remainder with STL, detect on the remainder with a robust threshold, and mark known events so they are not reported.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 3 MB). No account. |
| **You should know** | Seasonal baselines and known events, from [Normal changes with the hour and the season](https://learning.nextia-ai.com/courses/anomaly-detection/m05/normal-changes-with-the-hour-and-the-season/); the robust z-score, from [Median, MAD and IQR](https://learning.nextia-ai.com/courses/anomaly-detection/m02/median-mad-and-iqr/). The forecasting course explains decomposition in depth. |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m05/stl-residuals/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M05-L02-stl-residuals/stl-residuals-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2, statsmodels 0.15.0. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 statsmodels==0.15.0
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'statsmodels']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`orders.csv`, `calendar.csv`, `ticket_api_metrics.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "ticket_api_metrics.csv.gz": "73b420194586f72b760fd672f89ee2526035385965668a3098cf1e34dfa123c4",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Three parts of a series, by hand

A shop's daily orders mix three things: a slow level (the **trend**), a pattern that repeats every week (the **season**), and what is left (the **remainder**). An odd day hides in the remainder.

Run the next cell. It splits three made-up weeks by hand: a 7-day moving average for the trend, the average weekday effect for the season, and the rest. Week 3 has a low Thursday (500).

> **Check.** The low Thursday has a remainder of -80.0. Every other day is between -23.9 and 40.0.

In [ ]:
# Three weeks of a made-up shop: a level that rises slowly, a weekly pattern, and one odd day.
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 3
orders = np.array([600, 580, 590, 600, 640, 800, 760,
                   620, 600, 610, 620, 660, 820, 780,
                   640, 620, 630, 500, 680, 840, 800], dtype=float)   # week 3: Thursday is low
trend = pd.Series(orders).rolling(7, center=True, min_periods=4).mean()   # 1. the slow level
detrended = orders - trend
season = detrended.groupby(np.arange(21) % 7).transform("mean")           # 2. the usual weekday effect
remainder = orders - trend - season                                         # 3. what is left
print(pd.DataFrame({"day": days, "orders": orders, "trend": trend.round(1), "season": season.round(1),
                    "remainder": remainder.round(1)}).to_string(index=False))

The low Thursday has 500 orders. Against the mean of all 21 days, its z-score is only -1.76: the slow rise and the busy weekends make the spread large. In the remainder, it stands out at once. **STL** (Seasonal and Trend decomposition using Loess) does the same split, but better: it follows a trend that bends, a season that changes slowly, and with `robust=True` it is not pulled by odd days.

## 2. STL on Larkfield's daily orders

The next cell loads the two years of daily orders and runs STL with a weekly season (`period=7`). STL needs a value on every day, so the missing day (2025-10-19) is filled from its neighbours first. It prints the trend of one week in September 2024.

In [ ]:
from statsmodels.tsa.seasonal import STL

orders = pd.read_csv("data/orders.csv", parse_dates=["date"]).groupby("date").orders.sum()
calendar = pd.read_csv("data/calendar.csv", parse_dates=["date"]).set_index("date")
days = pd.date_range(orders.index.min(), orders.index.max(), freq="D")
orders = orders.reindex(days).astype(float)                     # 2025-10-19 is missing: NaN
holiday = calendar.is_holiday.reindex(days).fillna(0).astype(bool)
near_holiday = holiday | holiday.shift(1, fill_value=False) | holiday.shift(-1, fill_value=False)
after_warm_up = orders.index >= "2024-05-06"                     # the same days as the last lesson
filled = orders.interpolate(limit_direction="both")              # STL needs a value on every day
stl = STL(filled, period=7, robust=True).fit()
print(stl.trend.round(1).loc["2024-09-09":"2024-09-15"].to_string())

The chart shows the four parts. The trend carries the yearly season (the garden months, the quiet winter) and the slow growth. The weekly season is small. The remainder is what STL could not explain.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 7), sharex=True)
for ax, part, name in zip(axes, [orders, stl.trend, stl.seasonal, stl.resid], ["orders", "trend", "season", "remainder"]):
    ax.plot(part.index, part, lw=0.7, color="grey")
    ax.set_ylabel(name)
plt.tight_layout()
plt.show()

## 3. Detect on the remainder

Run the next cell. It computes the robust z-score of the remainder (median and MAD, as in Module 2), and alarms when it is above 3.5. Black Friday shows why: its orders are far above the trend, and the remainder carries almost all of it.

> **Check.** You should see 91 alarm days, 16 of them on or next to a holiday.

In [ ]:
def robust_z(x, center, spread):
    return (x - center) / (1.4826 * np.maximum(spread, 1e-9))

r = stl.resid
center, mad = r.median(), (r - r.median()).abs().median()
stl_alarm = (robust_z(r, center, mad).abs() > 3.5) & orders.notna()       # never on the missing day
a = stl_alarm & after_warm_up
print(f"limit: remainder beyond {3.5 * 1.4826 * mad:.1f} orders from {center:.1f}")
print("alarm days:", int(a.sum()), "| on or next to a holiday:", int((a & near_holiday).sum()))
print("Black Friday 2025:", {k: round(float(v.loc["2025-11-28"]), 1) for k, v in
                             {"orders": orders, "trend": stl.trend, "season": stl.seasonal, "remainder": r}.items()})

> **Predict.** The last lesson's seasonal baseline reported 32 days after the holidays were marked. Will STL report more or fewer?

In [ ]:
reported = a & ~near_holiday
print("reported after marking holidays:", int(reported.sum()))
print("the first ten:", [str(d.date()) for d in reported.index[reported]][:10])

STL reports **more** days (75), not fewer. On normal days the remainder is small, so the robust limit is narrow: 75.6 orders. Any day that the trend and the weekly season do not explain goes over it: the planted situations of the shared series, such as the heat wave, and many ordinary busy or quiet days. STL also looks at the days **after** each day: it is a tool to study history, not a live alert. The next lesson forecasts each day from the past only.

### A common mistake: STL and the missing day

We filled the missing day (2025-10-19) before STL. Run the next cell: STL on the series with the gap left as `NaN`.

In [ ]:
broken = STL(orders, period=7, robust=True).fit()           # the missing day left as NaN, not filled
print("days with no remainder:", int(broken.resid.isna().sum()), "of", len(orders))

STL gives **no error**. But the gap spreads: 386 of the 728 days have no remainder (`NaN`). A robust z of `NaN` is never above 3.5, so those days can never alarm. Fill a missing value before STL (here, from its neighbours), and keep the row count of each day as a separate check.

## 4. STL on the ticket API's latency

The ticket API has a planted memory leak (2025-09-22 03:00 to 2025-09-27 04:30: latency rises slowly until a restart) and a planted release spike (2025-10-08, 14:05 to 14:45). Run the next cell: STL on the hourly mean of the p95 latency, with a daily season (`period=24`), and the comparison-table row.

In [ ]:
import compare
metrics = pd.read_csv("data/ticket_api_metrics.csv.gz", parse_dates=["ts"]).set_index("ts")
hourly = metrics.latency_p95_ms.resample("1h").mean()            # 1,440 hours: 60 days
h_stl = STL(hourly, period=24, robust=True).fit()                 # the season is one day
hr = h_stl.resid
hz = robust_z(hr, hr.median(), (hr - hr.median()).abs().median())
h_alarm = hz.abs() > 4
row = compare.score_windows(hourly.index, h_alarm, "ticket_api", "STL residual, hourly latency",
                            "period 24 hours; |robust z| > 4", gap="2h", explain_minutes=3, signal="latency")
print({k: row[k] for k in ("alert events", "false alert events", "minutes to detection")})

The chart: the latency with the STL trend (top), and the robust z of the remainder with the limits ±4 (bottom). The orange band is the memory leak.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(hourly.index, hourly, lw=0.6, color="grey", label="p95 latency, hourly mean (ms)")
axes[0].plot(hourly.index, h_stl.trend, lw=1.5, color="black", label="STL trend")
axes[0].legend(loc="upper right")
axes[1].plot(hz.index, hz, lw=0.6, color="grey")
axes[1].axhline(4, ls="--", color="red"); axes[1].axhline(-4, ls="--", color="red")
axes[1].set_ylabel("robust z of the remainder")
for ax in axes:
    ax.axvspan(pd.Timestamp("2025-09-22 03:00"), pd.Timestamp("2025-09-27 04:30"), color="orange", alpha=0.2)
plt.tight_layout()
plt.show()

> **Predict.** The leak raises the latency for five days. Why does the STL remainder stay small for most of them?

In [ ]:
print("trend at the leak's start and end:", round(float(h_stl.trend.loc["2025-09-22 03:00"]), 1),
      round(float(h_stl.trend.loc["2025-09-27 04:00"]), 1), "ms")
print("largest |z| on the leak's first day:", round(float(hz.loc["2025-09-22 03:00":"2025-09-23 03:00"].abs().max()), 2))
print("release hour 2025-10-08 14:00, z:", round(float(hz.loc["2025-10-08 14:00"]), 2))

The **trend absorbs the leak**. A slow rise is exactly what a trend is for, so STL moves it into the trend, and the remainder stays normal. The first alarm inside the leak comes 6,000 minutes after it starts. The release spike is the opposite: one sudden hour, which no trend follows, so it is found at once (z 14.54). The restart at the end of the leak also gives a large remainder: the latency drops at once. A fix can look like an anomaly too.

## 5. Check yourself

**Your turn 1.** How many days does STL report on the daily orders after the warm-up, when the holidays and the days next to them are marked as known events? Write it as `stl_reported`.

In [ ]:
stl_reported = int((stl_alarm & after_warm_up & ~near_holiday).sum())

In [ ]:
expect('stl_reported', stl_reported, 75)

**Your turn 2.** At what hour does the hourly STL detector first alarm inside the memory leak (2025-09-22 03:00 to 2025-09-27 04:30)? Write it as a string like `"2025-09-22 03:00:00"` in `first_leak_alarm`.

In [ ]:
in_leak = (hourly.index >= "2025-09-22 03:00") & (hourly.index <= "2025-09-27 04:30")
first_leak_alarm = str(hourly.index[h_alarm & in_leak].min())

In [ ]:
expect_hash('first_leak_alarm', first_leak_alarm, '09ee8a4a02059535')

## Recap

- STL splits a series into trend, season and remainder. Detect on the remainder with a robust z-score.
- Mark known events (holidays, campaigns), or they fill the report. STL on daily orders still reports many days: every planted situation of the series is in its remainder.
- A slow change goes into the trend, so STL finds it late or never. A sudden spike stands out at once.
- STL uses the days after each day: use it to study history. A live alert needs a forecast from the past only.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Forecast and compare](https://learning.nextia-ai.com/courses/anomaly-detection/m05/forecast-and-compare/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m05/stl-residuals/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.